# Web Scraper

### Install and import libraries

In [ ]:
!pip bs4 requests

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import json
import time
import os
from typing import List, Dict, Set

### Define constraits

We will deploy a recursive crawler, that is, it not only scrapes the page given, it also recursively clicks links in the page and scrape them.

What websites to scrape from?

In [ ]:
SEED_URLS = [
    "https://innowings.engg.hku.hk/",
    "https://innoacademy.engg.hku.hk/",
]

Restrict recursive crawler to websites owned by Innowing.

In [ ]:
ALLOWED_DOMAINS = {"innowings.engg.hku.hk", "innoacademy.engg.hku.hk"}

# Check if a URL belongs to Innowing
def is_internal_link(url: str) -> bool:
    parsed = urlparse(url)
    return parsed.netloc in ALLOWED_DOMAINS or not parsed.netloc  # allow relative links

Other constraints

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/133.0 Safari/537.36"
}
MAX_PAGES = 150          # Safety limit - increase if needed
DELAY = 1.0              # Seconds between requests (polite enough for this small site)

### Define helper functions

In [ ]:
# Checks whether a link belongs to the allowed domains (prevents crawling unrelated websites)
def is_internal_link(url: str) -> bool:
    parsed = urlparse(url)
    return parsed.netloc in ALLOWED_DOMAINS or not parsed.netloc  # allow relative links

# Performs the most basic cleaning: removes only JavaScript and CSS, then extracts readable text while reducing extra blank lines.
def simple_clean_text(soup: BeautifulSoup) -> str:
    # Very basic cleaning - remove script/style only
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    # Remove excessive blank lines
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

# Downloads a single page and returns a dictionary containing the URL and its cleaned text (or an error message)
def scrape_page(url: str) -> Dict[str, str]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        text = simple_clean_text(soup)
        return {"url": url, "text": text}
    except Exception as e:
        print(f"❌ Failed {url}: {e}")
        return {"url": url, "text": f"[ERROR: {e}]"}

### Perform web crawling

In [ ]:
print("🚀 Starting crawling...\n")

visited: Set[str] = set()
queue: List[str] = SEED_URLS[:]
documents: List[Dict[str, str]] = []

while queue and len(documents) < MAX_PAGES:
    url = queue.pop(0)
    if url in visited:
        continue

    print(f"📄 [{len(documents)+1}/{MAX_PAGES}] Scraping → {url}")
    doc = scrape_page(url)
    
    if doc["text"].strip():
        documents.append(doc)
    
    visited.add(url)

    # Extract links (very basic)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.find_all("a", href=True):
            full_url = urljoin(url, a["href"])
            if is_internal_link(full_url) and full_url not in visited:
                queue.append(full_url)
    except:
        pass  # ignore link extraction errors

    time.sleep(DELAY)

print(f"\n✅ Crawl finished! Collected {len(documents)} pages.")

### Save the data to a file

In [ ]:
DATASET = os.getenv("DATASET") or "data.json"

# Save to data.json
with open(DATASET, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"💾 Saved to {DATASET} — ready for ingest.py")